# 01 — Classificatore a feature (TF-IDF + regressione logistica)

Primo passo della versione Web Science 2027: addestrare un classificatore semplice e leggibile sul dataset
così come esce dallo scraping, e misurare come va.

- **Dati:** gli articoli prodotti da `0-scraping/` (scraping + filtro della lingua).
- **Valutazione:** 5 fold separati per testata (nessuna testata sta sia nel training sia nel test), **identici a quelli del paper**.
- **Metriche:** accuracy, F1 macro e tolerant accuracy, sia per articolo sia per testata (voto di maggioranza).
- **Confronto:** gli stessi numeri per BERT, RoBERTa, Llama e Mistral, calcolati sulle predizioni del paper.

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

DATASET = "../Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion.csv"
PAPER_PREDICTIONS = "../../IrForGood/Results/derived/predictions_long.csv"
RESULTS = "../Results/"
SEED = 42
LIVELLI = ["l1", "l2", "l3", "l4", "l5"]   # etichette 0..4 nel dataset

## 1. I dati

Un articolo per riga. `label` è il livello NewsGuard della testata (0 = $l_1$, …, 4 = $l_5$).
L'`article_id` è la posizione della riga, lo stesso identificativo usato nel paper.

In [7]:
df = pd.read_csv(DATASET)
df["article_id"] = df.index

print(len(df), "articoli,", df.domain.nunique(), "testate")
df.groupby("label").agg(articoli=("url", "size"), testate=("domain", "nunique")).rename(index=lambda k: LIVELLI[k])

19476 articoli, 439 testate


,articoli,testate
label,,
l1,4871,152
l2,3016,69
l3,4620,95
l4,5041,88
l5,1928,35


## 2. I fold: separazione per testata, come nel paper

Ogni testata finisce tutta in un solo fold. Le testate sono distribuite nei 5 fold mantenendo in ognuno
le stesse proporzioni di livelli (`StratifiedKFold` sulle testate, seed 42): è esattamente la procedura dei notebook del paper.
Subito dopo controlliamo che i fold coincidano con quelli usati da BERT, RoBERTa, Llama e Mistral.

In [8]:
livello_testata = df.groupby("domain").label.agg(lambda x: x.mode()[0])   # testate in ordine alfabetico

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_testata = {}
for fold, (_, test_idx) in enumerate(skf.split(livello_testata.index, livello_testata.values), start=1):
    for testata in livello_testata.index[test_idx]:
        fold_testata[testata] = fold
df["fold"] = df.domain.map(fold_testata)

paper = pd.read_csv(PAPER_PREDICTIONS)
fold_paper = paper[paper.model == "BERT"].set_index("article_id").fold.sort_index()
assert (df.fold.values == fold_paper.values).all()
print("I fold coincidono con quelli del paper.")

df.groupby("fold").agg(articoli=("url", "size"), testate=("domain", "nunique"))

I fold coincidono con quelli del paper.


,articoli,testate
fold,,
1,3895,88
2,4047,88
3,3665,88
4,3868,88
5,4001,87


## 3. Il classificatore

- **TF-IDF:** ogni articolo diventa un vettore di parole e coppie di parole, ciascuna con un peso
  (alto se frequente nell'articolo ma rara nel resto). Il vocabolario si impara **solo sugli articoli di training** del fold.
- **Regressione logistica:** impara quali parole spingono verso ciascun livello. `class_weight="balanced"` compensa
  il fatto che i livelli hanno numeri di articoli diversi.

Stesse impostazioni della prova fatta dopo la submission a IR for Good.

In [4]:
df["pred"] = -1

for fold in range(1, 6):
    train = df[df.fold != fold]
    test = df[df.fold == fold]

    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True,
                            max_features=100_000, strip_accents="unicode")
    X_train = tfidf.fit_transform(train.text)
    X_test = tfidf.transform(test.text)

    clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    clf.fit(X_train, train.label)
    df.loc[test.index, "pred"] = clf.predict(X_test)
    print(f"fold {fold}: {len(train)} articoli di training, {len(test)} di test")

fold 1: 15581 articoli di training, 3895 di test


fold 2: 15429 articoli di training, 4047 di test


fold 3: 15811 articoli di training, 3665 di test


fold 4: 15608 articoli di training, 3868 di test


fold 5: 15475 articoli di training, 4001 di test


## 4. Le metriche

- **Accuracy:** quota di predizioni esatte.
- **F1 macro:** media dell'F1 dei 5 livelli, ognuno con lo stesso peso (non premia chi indovina solo i livelli più numerosi).
- **Tolerant accuracy:** sfrutta il fatto che i livelli sono ordinati. Una predizione esatta vale 1, un livello adiacente
  vale $\alpha$, tutto il resto 0. Con $\alpha = 1$ è la quota di predizioni sbagliate al massimo di un livello;
  con $\alpha = 0{,}5$ l'errore di un livello vale mezzo punto. Sono le due versioni riportate nel paper.

Per le testate si usa il **voto di maggioranza** dei loro articoli (in caso di pareggio vince il livello più basso, come nel paper).
Ogni metrica è calcolata fold per fold; riportiamo la media e la deviazione standard sui 5 fold.

In [5]:
def tolerant(y_true, y_pred, alpha):
    distanza = np.abs(np.asarray(y_true) - np.asarray(y_pred))
    return np.mean(np.where(distanza == 0, 1.0, np.where(distanza == 1, alpha, 0.0)))

def metriche(y_true, y_pred):
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "F1 macro": f1_score(y_true, y_pred, average="macro"),
            "Tolerant α=0.5": tolerant(y_true, y_pred, 0.5),
            "Tolerant α=1.0": tolerant(y_true, y_pred, 1.0)}

def voto(predizioni):
    conteggi = predizioni.value_counts()
    return conteggi[conteggi == conteggi.max()].index.min()

def per_testata(articoli):
    return articoli.groupby(["fold", "domain"], as_index=False).agg(
        y_true=("y_true", "first"), y_pred=("y_pred", voto), articoli=("y_pred", "size"))

def tabella_per_fold(dati):
    t = pd.DataFrame([{"fold": f, **metriche(g.y_true, g.y_pred)} for f, g in dati.groupby("fold")]).set_index("fold")
    media, std = t.mean(), t.std()
    t.loc["media"], t.loc["dev. std"] = media, std
    return t.round(3)

## 5. Risultati del classificatore a feature

In [6]:
art = df.rename(columns={"label": "y_true", "pred": "y_pred"})[["article_id", "domain", "fold", "y_true", "y_pred"]]
pub = per_testata(art)

print("PER ARTICOLO")
display(tabella_per_fold(art))
print("PER TESTATA (voto di maggioranza)")
display(tabella_per_fold(pub))

PER ARTICOLO


,Accuracy,F1 macro,Tolerant α=0.5,Tolerant α=1.0
fold,,,,
1,0.586,0.521,0.696,0.805
2,0.547,0.506,0.683,0.820
3,0.591,0.524,0.722,0.854
4,0.564,0.518,0.665,0.766
5,0.595,0.535,0.715,0.835
media,0.576,0.521,0.696,0.816
dev. std,0.021,0.010,0.023,0.033


PER TESTATA (voto di maggioranza)


,Accuracy,F1 macro,Tolerant α=0.5,Tolerant α=1.0
fold,,,,
1,0.625,0.498,0.733,0.841
2,0.682,0.557,0.773,0.864
3,0.693,0.585,0.795,0.898
4,0.705,0.587,0.801,0.898
5,0.701,0.595,0.799,0.897
media,0.681,0.564,0.780,0.879
dev. std,0.033,0.040,0.029,0.026


### Dove sbaglia, per testata

Righe = livello vero, colonne = livello predetto (numero di testate).

In [7]:
pd.crosstab(pub.y_true.map(lambda k: LIVELLI[k]), pub.y_pred.map(lambda k: LIVELLI[k]),
            rownames=["vero"], colnames=["predetto"])

predetto,l1,l2,l3,l4,l5
vero,,,,,
l1,135,9,4,4,0
l2,1,54,4,9,1
l3,0,19,57,17,2
l4,0,20,8,51,9
l5,0,9,4,20,2


In [8]:
accuratezza_livello = pub.assign(giusta=pub.y_true == pub.y_pred).groupby("y_true").giusta.agg(["sum", "size", "mean"])
accuratezza_livello.columns = ["testate giuste", "testate", "accuratezza"]
accuratezza_livello.rename(index=lambda k: LIVELLI[k]).round(3)

,testate giuste,testate,accuratezza
y_true,,,
l1,135,152,0.888
l2,54,69,0.783
l3,57,95,0.600
l4,51,88,0.580
l5,2,35,0.057


## 6. Confronto con i modelli del paper

Stessi fold, stesse metriche, stesso voto di maggioranza, applicati alle predizioni per articolo di BERT, RoBERTa,
Llama e Mistral salvate per il paper. Media sui 5 fold.

In [9]:
modelli = pd.concat([
    paper[["model", "domain", "fold", "y_true", "y_pred"]].astype({"model": str}),
    art.assign(model="TF-IDF + LR")[["model", "domain", "fold", "y_true", "y_pred"]],
])

righe_art, righe_pub = [], []
for modello in ["TF-IDF + LR", "BERT", "RoBERTa", "Llama", "Mistral"]:
    g = modelli[modelli.model == modello]
    righe_art.append({"modello": modello, **tabella_per_fold(g).loc["media"]})
    righe_pub.append({"modello": modello, **tabella_per_fold(per_testata(g)).loc["media"]})

print("PER ARTICOLO (media sui fold)")
display(pd.DataFrame(righe_art).set_index("modello"))
print("PER TESTATA (media sui fold)")
display(pd.DataFrame(righe_pub).set_index("modello"))

PER ARTICOLO (media sui fold)


,Accuracy,F1 macro,Tolerant α=0.5,Tolerant α=1.0
modello,,,,
TF-IDF + LR,0.576,0.521,0.696,0.816
BERT,0.583,0.517,0.702,0.821
RoBERTa,0.603,0.539,0.717,0.830
Llama,0.599,0.548,0.718,0.838
Mistral,0.611,0.544,0.723,0.835


PER TESTATA (media sui fold)


,Accuracy,F1 macro,Tolerant α=0.5,Tolerant α=1.0
modello,,,,
TF-IDF + LR,0.681,0.564,0.780,0.879
BERT,0.665,0.550,0.768,0.870
RoBERTa,0.690,0.604,0.783,0.875
Llama,0.665,0.575,0.772,0.879
Mistral,0.683,0.592,0.770,0.856


## 7. Salvataggio

Le predizioni contengono le etichette NewsGuard: restano nel repository privato e non vanno pubblicate.

In [10]:
art.to_csv(RESULTS + "01_tfidf_predizioni_articoli.csv", index=False)
pub.to_csv(RESULTS + "01_tfidf_predizioni_testate.csv", index=False)
pd.concat({"articoli": tabella_per_fold(art), "testate": tabella_per_fold(pub)}).to_csv(RESULTS + "01_tfidf_metriche.csv")
print("Salvato in", RESULTS)

Salvato in ../Results/
